# Redes neuronales y texto tokenizado — AP_2026Q2

Este notebook agrega tres familias reproducibles: MLP tabular, MLP con SVD de TF-IDF y MLP sobre una representación combinada de tokens de palabras y caracteres. Se evalúan con el mismo holdout agrupado que CatBoost.

La tokenización se ajusta exclusivamente sobre train. No se usa `sold_quantity` ni información posterior a la publicación sin auditar, para evitar leakage.

## Idea conceptual

`texto → tokens → TF-IDF → SVD → red densa → probabilidad de used`

La red aprende combinaciones no lineales entre señales numéricas, categorías codificadas y componentes semánticos. En este dataset, los tokens de palabra capturan términos como *nuevo*, *original* o *usado*; los tokens de carácter capturan códigos, modelos, talles y errores ortográficos.

In [ ]:
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


In [ ]:
# Tokenización robusta: word + character n-grams
word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_features=120_000, sublinear_tf=True)
char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=3, max_features=120_000, sublinear_tf=True)
# Ajustar ambos únicamente con train/dev; transformar holdout y test sin refit.
# text_matrix = sparse.hstack([word.fit_transform(title), char.fit_transform(title)])
# svd = TruncatedSVD(n_components=128, random_state=42)
# text_dense = svd.fit_transform(text_matrix)


## Modelo 1 — MLP tabular

Dos capas ocultas permiten aprender interacciones suaves. Ventaja: puede combinar muchas variables escaladas. Desventajas: es sensible a escalado, nulos y alta dimensionalidad; suele perder frente a CatBoost cuando hay categorías de alta cardinalidad.

In [ ]:
mlp_tabular = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(256,128), activation='relu', alpha=1e-4,
                  batch_size=512, learning_rate_init=1e-3, early_stopping=True,
                  validation_fraction=.15, max_iter=80, random_state=42)
)
# mlp_tabular.fit(X_dev, y_dev)


## Modelo 2 — MLP texto SVD + tabular

Se concatenan las variables tabulares robustas con los componentes SVD. Hiperparámetros principales: `hidden_layer_sizes` controla capacidad, `alpha` regulariza pesos, `batch_size` estabiliza el gradiente y `early_stopping` limita sobreajuste.

In [ ]:
mlp_text = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(384,192), activation='relu', alpha=3e-4,
                  batch_size=512, learning_rate_init=5e-4, early_stopping=True,
                  validation_fraction=.15, max_iter=100, random_state=42)
)
# X_dev_nn = np.hstack([X_tab_dev, text_dense_dev])
# mlp_text.fit(X_dev_nn, y_dev)


## Modelo 3 — MLP con representación word + char

Los n-gramas de caracteres son especialmente útiles para publicaciones en español con marcas, códigos, números y faltas de ortografía. Se recomienda comparar `n_components=64,128,192` y conservar el modelo solo si mejora el holdout; no se debe escoger por accuracy de entrenamiento.

In [ ]:
# Evaluación común
def report_model(name, model, Xh, yh):
    p = model.predict(Xh)
    return {'modelo':name, 'accuracy':accuracy_score(yh,p),
            'balanced_accuracy':balanced_accuracy_score(yh,p),
            'macro_f1':f1_score(yh,p,average='macro')}
# results = pd.DataFrame([...]).sort_values('accuracy', ascending=False)


## Reglas de decisión

El umbral se puede explorar entre 0,47 y 0,53, pero debe fijarse en el holdout documentado y mantenerse igual para generar test. No se incorporan reglas de categoría posteriores: la submission `deep_category_prior` empató en Kaggle con 0,93695 y no generalizó.

In [ ]:
# Submission compatible con la competencia
def make_submission(pred, path):
    out = pd.DataFrame({'ID': np.arange(1, len(pred)+1), 'condition': pred})
    assert len(out) == 30000 and out.ID.is_unique
    assert set(out.condition.unique()) <= {'new','used'}
    out.to_csv(path, index=False)
    return out.head()
